In [2]:
from pathlib import Path
import polars as pl

In [3]:
SILVER_DIR = (
    Path.cwd().parent.parent
    / "data"
    / "silver"
    / "static"
    / "latest"
)

assert SILVER_DIR.is_dir(), f"Silver dataset not found: {SILVER_DIR}"

print(SILVER_DIR.resolve())

/home/user/playground/transit-intelligence/data/silver/static/gtfs_fp2026_20260805


In [4]:
def load_parquet_tree(
    root: Path
) -> dict[str, pl.LazyFrame]:
    """
    Recursively discover all Parquet files under `root` and return them as Polars Lazyframes

    Keys are paths relative to `root`, without the .parquet suffix.
    """
    datasets: dict[str, pl.LazyFrame] = {}

    for path in sorted(root.rglob("*.parquet")):
        key = str(path.relative_to(root).with_suffix(""))

        datasets[key] = pl.scan_parquet(path)

    return datasets

In [5]:
datasets = load_parquet_tree(SILVER_DIR)

for name, df in datasets.items():
    print(name)

agencies
calendar
calendar_dates
crossing_routes
crossing_stop_times
crossing_trips
frequencies
internal_routes
internal_stop_times
internal_trips
mixed_routes
routes
stop_times
stops
trip_ids
trips


In [6]:
def profile_datasets(
    datasets: dict[str, pl.LazyFrame],
) -> pl.DataFrame:
    rows = []

    for name, lf in datasets.items():
        rows.append(
            {
                "dataset": name,
                "columns": len(lf.collect_schema()),
                "rows": lf.select(pl.len()).collect().item(),
            }
        )

    return (
        pl.DataFrame(rows)
        .sort("dataset")
    )

In [7]:
profile = profile_datasets(datasets)
profile

dataset,columns,rows
str,i64,i64
"""agencies""",6,19
"""calendar""",10,15518
"""calendar_dates""",3,2236321
"""crossing_routes""",6,171
"""crossing_stop_times""",7,1684359
…,…,…
"""routes""",6,264
"""stop_times""",7,3146340
"""stops""",9,2002


In [8]:
for name, lf in datasets.items():
    print(f"\n{'=' * 60}")
    print(name)
    print('=' * 60)
    print(lf.collect_schema())


agencies
Schema({'agency_id': String, 'agency_name': String, 'agency_url': String, 'agency_timezone': String, 'agency_lang': String, 'agency_phone': String})

calendar
Schema({'service_id': String, 'monday': String, 'tuesday': String, 'wednesday': String, 'thursday': String, 'friday': String, 'saturday': String, 'sunday': String, 'start_date': String, 'end_date': String})

calendar_dates
Schema({'service_id': String, 'date': String, 'exception_type': String})

crossing_routes
Schema({'route_id': String, 'agency_id': String, 'route_short_name': String, 'route_long_name': String, 'route_desc': String, 'route_type': String})

crossing_stop_times
Schema({'trip_id': String, 'arrival_time': String, 'departure_time': String, 'stop_id': String, 'stop_sequence': String, 'pickup_type': String, 'drop_off_type': String})

crossing_trips
Schema({'route_id': String, 'service_id': String, 'trip_id': String, 'trip_headsign': String, 'trip_short_name': String, 'direction_id': String, 'block_id': Strin

In [9]:
def cardinality(lf: pl.LazyFrame, column: str) -> dict:
    result = (
        lf.select(
            pl.len().alias("rows"),
            pl.col(column).n_unique().alias("unique"),
        )
        .collect()
        .row(0, named=True)
    )

    return result

In [13]:
def missing_references(
    child: pl.LazyFrame,
    child_key: str,
    parent: pl.LazyFrame,
    parent_key: str,
) -> pl.DataFrame:
    return (
        child
        .join(
            parent.select(parent_key).unique(),
            left_on=child_key,
            right_on=parent_key,
            how="anti",
        )
        .select(child_key)
        .unique()
        .collect()
    )

In [14]:
checks = {
    "routes → agencies": (
        datasets["routes"],
        "agency_id",
        datasets["agencies"],
        "agency_id",
    ),
    "trips → routes": (
        datasets["trips"],
        "route_id",
        datasets["routes"],
        "route_id",
    ),
    "stop_times → trips": (
        datasets["stop_times"],
        "trip_id",
        datasets["trips"],
        "trip_id",
    ),
    "stop_times → stops": (
        datasets["stop_times"],
        "stop_id",
        datasets["stops"],
        "stop_id",
    ),
}

In [15]:
for relationship, args in checks.items():
    missing = missing_references(*args)

    print(
        f"{relationship}: "
        f"{missing.height:,} missing references"
    )

routes → agencies: 0 missing references
trips → routes: 0 missing references
stop_times → trips: 0 missing references
stop_times → stops: 2,250 missing references


In [17]:
missing_stop_times = (
    datasets["stop_times"]
    .join(
        datasets["stops"].select("stop_id").unique(),
        on="stop_id",
        how="anti",
    )
    .collect()
)

missing_stop_times.shape

(1000573, 7)

In [18]:
missing_stop_times.select(
    "stop_id"
).unique().sort("stop_id")

stop_id
str
"""8011068"""
"""8014010"""
"""8014021"""
"""8014309"""
"""8014325"""
…
"""missingSLOID_5457076"""
"""missingSLOID_8001093"""
"""missingSLOID_8014008"""


In [19]:
missing_stop_times.select(
    "trip_id",
    "stop_id",
    "stop_sequence",
).sort(
    ["trip_id", "stop_sequence"]
).head(100)

trip_id,stop_id,stop_sequence
str,str,str
""".ojp-91-1-D.1.TA.1000.j26""","""ch:1:sloid:1026:2:4""","""1"""
""".ojp-91-1-D.1.TA.1000.j26""","""ch:1:sloid:6000:3:5""","""11"""
""".ojp-91-1-D.1.TA.1000.j26""","""ch:1:sloid:6206:1:1""","""12"""
""".ojp-91-1-D.1.TA.1000.j26""","""ch:1:sloid:6208:2:4""","""13"""
""".ojp-91-1-D.1.TA.1000.j26""","""ch:1:sloid:6209:1:2""","""14"""
…,…,…
""".ojp-91-1-D.1.TA.1007.j26""","""ch:1:sloid:7000:0:229097""","""6"""
""".ojp-91-1-D.1.TA.1008.j26""","""ch:1:sloid:1026:2:4""","""1"""
""".ojp-91-1-D.1.TA.1008.j26""","""ch:1:sloid:6000:3:5""","""11"""


In [20]:
missing_stop_times.select(
    pl.len().alias("missing_stop_time_rows"),
    pl.col("stop_id").n_unique().alias("missing_stop_ids"),
    pl.col("trip_id").n_unique().alias("affected_trips"),
)

missing_stop_time_rows,missing_stop_ids,affected_trips
u32,u32,u32
1000573,2250,112854


In [22]:
internal_affected = (
    datasets["internal_trips"]
    .select("trip_id")
    .join(
        missing_stop_times.lazy().select("trip_id").unique(),
        on="trip_id",
        how="inner",
    )
    .select(pl.col("trip_id").n_unique())
    .collect()
    .item()
)

crossing_affected = (
    datasets["crossing_trips"]
    .select("trip_id")
    .join(
        missing_stop_times.lazy().select("trip_id").unique(),
        on="trip_id",
        how="inner",
    )
    .select(pl.col("trip_id").n_unique())
    .collect()
    .item()
)

print(f"Internal affected trips:  {internal_affected:,}")
print(f"Crossing affected trips:  {crossing_affected:,}")

Internal affected trips:  0
Crossing affected trips:  112,854


In [26]:
print(
    "Referential integrity observation: \n"
    "crossing trips retain stop_times for external stops; \n"
    "these stops are intentionally absent from the Zurich-scoped stops dataset."
)

Referential integrity observation: 
crossing trips retain stop_times for external stops; 
these stops are intentionally absent from the Zurich-scoped stops dataset.


In [31]:
stops_ids = (
    datasets["stops"]
    .select("stop_id")
    .collect()
    .get_column("stop_id")
)

type(stops_ids), stops_ids.len()

crossing_boundaries = (
    datasets["crossing_stop_times"]
    .sort(["trip_id", "stop_sequence"])
    .group_by("trip_id", maintain_order=True)
    .agg(
        pl.col("stop_id").first().alias("first_stop_id"),
        pl.col("stop_id").last().alias("last_stop_id"),
    )
    .with_columns(
        pl.col("first_stop_id")
        .is_in(stops_ids)
        .alias("first_is_zurich"),

        pl.col("last_stop_id")
        .is_in(stops_ids)
        .alias("last_is_zurich"),
    )
    .collect()
)

/tmp/ipykernel_32776/4131011715.py:27: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .collect()


In [32]:
crossing_boundaries.group_by(
    ["first_is_zurich", "last_is_zurich"]
).len()

first_is_zurich,last_is_zurich,len
bool,bool,u32
true,true,12814
false,true,37291
false,false,26133
true,false,36616


In [33]:
stops_id = (
    datasets["stops"]
    .select("stop_id")
    .collect()
    .get_column("stop_id")
)

crossing_sequence = (
    datasets["crossing_stop_times"]
    .sort(["trip_id", "stop_sequence"])
    .with_columns(
        pl.col("stop_id")
        .is_in(stops_id, nulls_equal=False)
        .alias("is_zurich")
    )
    .with_columns(
        pl.col("is_zurich")
        .shift(1)
        .over("trip_id")
        .alias("previous_is_zurich")
    )
)

In [34]:
boundary_transitions = (
    crossing_sequence
    .filter(
        pl.col("previous_is_zurich").is_not_null()
        & (
            pl.col("is_zurich")
            != pl.col("previous_is_zurich")
        )
    )
)

In [36]:
transition_counts = (
    boundary_transitions
    .group_by("trip_id")
    .agg(
        pl.len().alias("boundary_transitions")
    )
    .group_by("boundary_transitions")
    .agg(
        pl.len().alias("trip_count")
    )
    .sort("boundary_transitions")
    .collect()
)

transition_counts

/tmp/ipykernel_32776/2647000843.py:12: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .collect()


boundary_transitions,trip_count
u32,u32
1,38288
2,31539
3,28572
4,7356
5,4330
6,52
7,2687
9,30


In [37]:
boundary_transitions.select(
    "trip_id",
    "stop_sequence",
    "stop_id",
    "previous_is_zurich",
    "is_zurich",
).head(50)

In [40]:
seven_transition_trips = (
    boundary_transitions
    .group_by("trip_id")
    .agg(
        pl.len().alias("boundary_transitions")
    )
    .filter(
        pl.col("boundary_transitions") == 7
    )
    .select("trip_id")
    .collect()
)

example_trip_id = seven_transition_trips.get_column("trip_id")[0]

example_trip_id

/tmp/ipykernel_32776/790643480.py:11: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .collect()


'.ojp-91-2-H.1.TA.1000.j26'

In [41]:
example_trip = (
    crossing_sequence
    .filter(
        pl.col("trip_id") == example_trip_id
    )
    .select(
        "trip_id",
        "stop_sequence",
        "stop_id",
        "is_zurich",
    )
    .sort("stop_sequence")
    .collect()
)

/tmp/ipykernel_32776/3658079910.py:13: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .collect()


In [42]:
example_trip = (
    example_trip
    .join(
        datasets["stops"]
        .select(["stop_id", "stop_name"])
        .collect(),
        on="stop_id",
        how="left",
    )
    .select(
        "stop_sequence",
        "stop_id",
        "stop_name",
        "is_zurich",
    )
)

example_trip

stop_sequence,stop_id,stop_name,is_zurich
str,str,str,bool
"""1""","""ch:1:sloid:96007::0""",null,false
"""10""","""ch:1:sloid:91165::0""","""Zürich, Grimselstrasse""",true
"""11""","""ch:1:sloid:91222::0""","""Zürich, Kappeli""",true
"""12""","""ch:1:sloid:91138::0""","""Zürich, Freihofstrasse""",true
"""13""","""ch:1:sloid:91252::0""","""Zürich, Letzigrund""",true
…,…,…,…
"""5""","""ch:1:sloid:90795::0""",null,false
"""6""","""ch:1:sloid:91274::0""","""Zürich, Micafil""",true
"""7""","""ch:1:sloid:90318::2""","""Zürich, Farbhof""",true


In [44]:
crossing_sequence = (
    datasets["crossing_stop_times"]
    .with_columns(
        pl.col("stop_sequence").cast(pl.Int64),
        pl.col("stop_id").is_in(stops_id).alias("is_zurich"),
    )
    .sort(["trip_id", "stop_sequence"])
    .with_columns(
        pl.col("is_zurich")
        .shift(1)
        .over("trip_id")
        .alias("previous_is_zurich")
    )
)

In [45]:
BRONZE_DIR = (
    Path.cwd().parent.parent
    / "data"
    / "bronze"
    / "static"
    / "latest"
)

In [46]:
bronze_stops = pl.scan_parquet(
    BRONZE_DIR / "stops.parquet"
)

In [47]:
bronze_stops.collect_schema()

Schema([('stop_id', String),
        ('stop_name', String),
        ('stop_lat', String),
        ('stop_lon', String),
        ('location_type', String),
        ('parent_station', String),
        ('platform_code', String),
        ('original_stop_id', String),
        ('didok', String)])

In [48]:
example_trip = (
    crossing_sequence
    .filter(
        pl.col("trip_id") == example_trip_id
    )
    .select(
        "stop_sequence",
        "stop_id",
        "is_zurich",
    )
    .collect()
    .join(
        bronze_stops
        .select([
            "stop_id",
            "stop_name",
            "stop_lat",
            "stop_lon",
        ])
        .collect(),
        on="stop_id",
        how="left",
    )
    .sort("stop_sequence")
)

example_trip

/tmp/ipykernel_32776/2661365264.py:11: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .collect()


stop_sequence,stop_id,is_zurich,stop_name,stop_lat,stop_lon
i64,str,bool,str,str,str
1,"""ch:1:sloid:96007::0""",false,"""Schlieren, Geissweid""","""47.39789360""","""8.44483741"""
2,"""ch:1:sloid:90805::0""",false,"""Schlieren, Zentrum/Bahnhof""","""47.39776590""","""8.44872711"""
3,"""ch:1:sloid:90803::0""",false,"""Schlieren, Wagonsfabrik""","""47.39847734""","""8.45463803"""
4,"""ch:1:sloid:90791::0""",false,"""Schlieren, Gasometerbrücke""","""47.39760172""","""8.46054894"""
5,"""ch:1:sloid:90795::0""",false,"""Schlieren, Mülligen""","""47.39578962""","""8.46704376"""
…,…,…,…,…,…
26,"""ch:1:sloid:30813::0""",true,"""Zürich, Kreuzplatz""","""47.36478582""","""8.55424323"""
27,"""ch:1:sloid:91119::0""",true,"""Zürich, Englischviertelstrasse""","""47.36649555""","""8.55767479"""
28,"""ch:1:sloid:3083::2""",true,"""Zürich, Römerhof""","""47.36770024""","""8.56028889"""


In [49]:
example_trip = (
    crossing_sequence
    .filter(
        pl.col("trip_id") == example_trip_id
    )
    .select(
        "trip_id",
        "stop_sequence",
        "stop_id",
        "is_zurich",
        "previous_is_zurich",
    )
    .collect()
    .join(
        bronze_stops
        .select([
            "stop_id",
            "stop_name",
            "stop_lat",
            "stop_lon",
        ])
        .collect(),
        on="stop_id",
        how="left",
    )
    .with_columns(
        pl.col("is_zurich")
        .alias("is_in_zurich"),

        (
            pl.col("is_zurich")
            != pl.col("previous_is_zurich")
        )
        .fill_null(False)
        .alias("is_transition_edge"),
    )
    .select([
        "trip_id",
        "stop_sequence",
        "stop_id",
        "stop_name",
        "stop_lat",
        "stop_lon",
        "is_in_zurich",
        "is_transition_edge",
    ])
    .sort("stop_sequence")
)

/tmp/ipykernel_32776/3857131058.py:13: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .collect()


In [50]:
example_trip

trip_id,stop_sequence,stop_id,stop_name,stop_lat,stop_lon,is_in_zurich,is_transition_edge
str,i64,str,str,str,str,bool,bool
""".ojp-91-2-H.1.TA.1000.j26""",1,"""ch:1:sloid:96007::0""","""Schlieren, Geissweid""","""47.39789360""","""8.44483741""",false,false
""".ojp-91-2-H.1.TA.1000.j26""",2,"""ch:1:sloid:90805::0""","""Schlieren, Zentrum/Bahnhof""","""47.39776590""","""8.44872711""",false,false
""".ojp-91-2-H.1.TA.1000.j26""",3,"""ch:1:sloid:90803::0""","""Schlieren, Wagonsfabrik""","""47.39847734""","""8.45463803""",false,false
""".ojp-91-2-H.1.TA.1000.j26""",4,"""ch:1:sloid:90791::0""","""Schlieren, Gasometerbrücke""","""47.39760172""","""8.46054894""",false,false
""".ojp-91-2-H.1.TA.1000.j26""",5,"""ch:1:sloid:90795::0""","""Schlieren, Mülligen""","""47.39578962""","""8.46704376""",false,false
…,…,…,…,…,…,…,…
""".ojp-91-2-H.1.TA.1000.j26""",26,"""ch:1:sloid:30813::0""","""Zürich, Kreuzplatz""","""47.36478582""","""8.55424323""",true,false
""".ojp-91-2-H.1.TA.1000.j26""",27,"""ch:1:sloid:91119::0""","""Zürich, Englischviertelstrasse""","""47.36649555""","""8.55767479""",true,false
""".ojp-91-2-H.1.TA.1000.j26""",28,"""ch:1:sloid:3083::2""","""Zürich, Römerhof""","""47.36770024""","""8.56028889""",true,false


In [51]:
# Find trips with exactly 7 Zurich-scope boundary transitions
seven_transition_trips = (
    boundary_transitions
    .group_by("trip_id")
    .agg(
        pl.len().alias("boundary_transitions")
    )
    .filter(
        pl.col("boundary_transitions") == 7
    )
    .select("trip_id")
    .collect()
)

# Take the first such trip
example_trip_id = (
    seven_transition_trips
    .get_column("trip_id")
    .first()
)

example_trip_id

/tmp/ipykernel_32776/3101519326.py:12: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .collect()


'.ojp-91-2-H.1.TA.1000.j26'

In [52]:
silver_stop_ids = (
    datasets["stops"]
    .select("stop_id")
    .collect()
    .get_column("stop_id")
)

example_trip = (
    datasets["crossing_stop_times"]
    .filter(
        pl.col("trip_id") == example_trip_id
    )
    .with_columns(
        pl.col("stop_sequence").cast(pl.Int64)
    )
    .sort("stop_sequence")
    .select([
        "trip_id",
        "stop_sequence",
        "stop_id",
    ])
    .collect()
    .join(
        bronze_stops
        .select([
            "stop_id",
            "stop_name",
            "stop_lat",
            "stop_lon",
        ])
        .collect(),
        on="stop_id",
        how="left",
    )
    .with_columns(
        pl.col("stop_id")
        .is_in(silver_stop_ids)
        .alias("is_in_zurich"),
    )
    .with_columns(
        (
            pl.col("is_in_zurich")
            != pl.col("is_in_zurich").shift(1)
        )
        .fill_null(False)
        .alias("is_transition_edge"),
    )
)

/tmp/ipykernel_32776/2124364340.py:35: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .with_columns(


In [54]:
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(100)

example_trip

trip_id,stop_sequence,stop_id,stop_name,stop_lat,stop_lon,is_in_zurich,is_transition_edge
str,i64,str,str,str,str,bool,bool
""".ojp-91-2-H.1.TA.1000.j26""",1,"""ch:1:sloid:96007::0""","""Schlieren, Geissweid""","""47.39789360""","""8.44483741""",false,false
""".ojp-91-2-H.1.TA.1000.j26""",2,"""ch:1:sloid:90805::0""","""Schlieren, Zentrum/Bahnhof""","""47.39776590""","""8.44872711""",false,false
""".ojp-91-2-H.1.TA.1000.j26""",3,"""ch:1:sloid:90803::0""","""Schlieren, Wagonsfabrik""","""47.39847734""","""8.45463803""",false,false
""".ojp-91-2-H.1.TA.1000.j26""",4,"""ch:1:sloid:90791::0""","""Schlieren, Gasometerbrücke""","""47.39760172""","""8.46054894""",false,false
""".ojp-91-2-H.1.TA.1000.j26""",5,"""ch:1:sloid:90795::0""","""Schlieren, Mülligen""","""47.39578962""","""8.46704376""",false,false
""".ojp-91-2-H.1.TA.1000.j26""",6,"""ch:1:sloid:91274::0""","""Zürich, Micafil""","""47.39289501""","""8.47482317""",true,true
""".ojp-91-2-H.1.TA.1000.j26""",7,"""ch:1:sloid:90318::2""","""Zürich, Farbhof""","""47.39105842""","""8.47874881""",true,false
""".ojp-91-2-H.1.TA.1000.j26""",8,"""ch:1:sloid:91051::0""","""Zürich, Bachmattstrasse""","""47.38899067""","""8.48299784""",true,false
""".ojp-91-2-H.1.TA.1000.j26""",9,"""ch:1:sloid:91258::0""","""Zürich, Lindenplatz""","""47.38778038""","""8.48634856""",true,false
